# Epsilon Fund - BB Portfolio + BTC Regime Filter
Copy of `portfolio_bb.ipynb` with a BTC regime filter overlay.

**Regime filter logic:**  
- Regime 0 (Bull) → trade normally  
- Regime 1 (Recovery/Chop) → trade normally  
- Regime 2 (Bear) → flat — no new entries, exit existing positions  
- Regime 3 (Extreme Bear) → flat  

The BTC regime is used as a market-wide filter for all coins.  
Source: `topics/regime-classifier/data/predictions/btc_regime_predictions.parquet`

---

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import numpy.core
import io, contextlib

# Compatibility shim: pkl files were saved with numpy 2.x (uses numpy._core.*)
# but this environment has numpy 1.x (only numpy.core.* exists).
sys.modules.setdefault('numpy._core', numpy.core)
for _sub in ['numeric', 'multiarray', 'umath', 'fromnumeric', 'function_base',
             'arrayprint', 'shape_base']:
    try:
        sys.modules.setdefault(
            f'numpy._core.{_sub}',
            __import__(f'numpy.core.{_sub}', fromlist=[_sub])
        )
    except ImportError:
        pass

ROOT   = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'
WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'bb_breakout_wf', 'oos')
PREDS  = os.path.join(ROOT, 'topics', 'regime-classifier', 'data', 'predictions', 'btc_regime_predictions.parquet')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos, plot_closed_trade_equity
from wf_engine import cost_stress_test

client = get_binance_client()

---
## Load Data

In [ ]:
# ── load daily OOS pkl files ──────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded OOS pkls: {list(coin_dfs.keys())}')
for coin, df in coin_dfs.items():
    print(f'  {coin}: {df.index[0].date()} → {df.index[-1].date()}  cols={df.columns.tolist()}')

In [ ]:
# ── load regime predictions ───────────────────────────────────────────────────
regime_df = pd.read_parquet(PREDS)
regime_df.index = pd.to_datetime(regime_df.index).tz_localize(None)

print(f'Regime predictions: {regime_df.index[0].date()} → {regime_df.index[-1].date()}  ({len(regime_df)} rows)')
print(f'Columns: {regime_df.columns.tolist()}')
print()
print('Predicted regime distribution:')
print(regime_df['pred_regime'].value_counts().sort_index())
print()
print('Regime key: 0=Bull  1=Recovery/Chop  2=Bear  3=Extreme Bear')

---
## Apply Regime Filter

For each coin, zero out the position on days where BTC predicted regime is Bear (2) or Extreme Bear (3).  
The BTC regime acts as a market-wide risk filter — when BTC is in a bear regime, all alt positions are closed.

In [ ]:
def apply_regime_filter(coin_dfs, regime_df, allow_regimes=(0, 1)):
    """
    Zero out position on days where BTC pred_regime is not in allow_regimes.
    allow_regimes=(0,1) → trade during Bull + Recovery/Chop, flat during Bear/Extreme Bear
    allow_regimes=(0,)  → trade only during Bull (most conservative)
    """
    # Build daily mask: 1 = allow trading, 0 = flat
    mask = regime_df['pred_regime'].isin(allow_regimes).astype(int)
    mask.index = pd.to_datetime(mask.index).normalize()

    filtered = {}
    for coin, df in coin_dfs.items():
        d = df.copy()
        idx = pd.to_datetime(d.index).normalize()
        coin_mask = mask.reindex(idx).fillna(1).values  # default allow if no regime data
        d['position']      = (d['position']      * coin_mask).astype(int)
        d['position_size'] = d['position_size']  * coin_mask
        filtered[coin] = d

    return filtered


# Bull + Recovery/Chop → trade | Bear + Extreme Bear → flat
coin_dfs_filtered    = apply_regime_filter(coin_dfs, regime_df, allow_regimes=(0, 1))

# Bull only (more aggressive filter)
coin_dfs_bull_only   = apply_regime_filter(coin_dfs, regime_df, allow_regimes=(0,))

# Coverage check
print('Regime filter coverage per coin:')
print(f'  {"Coin":<6} {"Total days":>11} {"Regime avail":>13} {"Filtered out":>13} {"% flat":>8}')
for coin, df in coin_dfs.items():
    idx  = pd.to_datetime(df.index).normalize()
    mask = regime_df['pred_regime'].reindex(idx)
    avail = mask.notna().sum()
    flat  = mask.isin([2, 3]).sum()
    print(f'  {coin:<6} {len(df):>11} {avail:>13} {flat:>13} {flat/avail*100:>7.1f}%')

---
## Scenario

Set `SCENARIO = 'close'` to match the baseline notebook.

In [ ]:
SHOW_COINS = ['AVAX', 'BTC', 'ETH', 'NEAR', 'ADA']
COST       = 0.001

---
## Unfiltered Baseline (original portfolio)

In [ ]:
print('=' * 60)
print('UNFILTERED BASELINE')
print('=' * 60)
metrics_base = plot_portfolio_oos(
    coin_dfs   = coin_dfs,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},
    show       = True,
    cost       = COST,
    save_html  = None,
)

---
## Filtered: Bull + Recovery/Chop only

In [ ]:
print('=' * 60)
print('REGIME FILTER: Bull + Recovery/Chop  (flat during Bear / Extreme Bear)')
print('=' * 60)
metrics_filt = plot_portfolio_oos(
    coin_dfs   = coin_dfs_filtered,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},
    show       = True,
    cost       = COST,
    save_html  = None,
)

---
## Filtered: Bull only (most conservative)

In [ ]:
print('=' * 60)
print('REGIME FILTER: Bull only  (flat during all non-Bull regimes)')
print('=' * 60)
metrics_bull = plot_portfolio_oos(
    coin_dfs   = coin_dfs_bull_only,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},
    show       = True,
    cost       = COST,
    save_html  = None,
)

---
## Side-by-Side Summary

In [ ]:
def calmar(m):
    return m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

rows = [
    ('Unfiltered baseline',          metrics_base),
    ('Regime filter (Bull+Chop)',     metrics_filt),
    ('Regime filter (Bull only)',     metrics_bull),
]

print(f'  {"Strategy":<32}  {"Sharpe":>7}  {"Return":>9}  {"Max DD":>8}  {"Calmar":>7}  {"PF":>6}  {"WR":>7}')
print('  ' + '-' * 80)
for label, m in rows:
    c = calmar(m)
    print(f'  {label:<32}  {m["sharpe_ratio"]:>7.2f}  '
          f'{m["total_return"]*100:>8.1f}%  '
          f'{m["max_drawdown"]*100:>7.1f}%  '
          f'{c:>7.2f}  '
          f'{m["profit_factor"]:>6.2f}  '
          f'{m["win_rate"]*100:>6.1f}%')

---
## Cost Stress Test — Filtered Portfolio

In [ ]:
print('Cost stress test — Bull+Chop filter:')
print(f'  {"Cost":>8} {"Mult":>6} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Calmar":>8}')
print(f'  {"─"*8} {"─"*6} {"─"*8} {"─"*10} {"─"*10} {"─"*8}')

for mult in (1.0, 1.5, 2.0, 3.0):
    c = COST * mult
    with contextlib.redirect_stdout(io.StringIO()):
        m = plot_portfolio_oos(
            coin_dfs   = coin_dfs_filtered,
            show_coins = SHOW_COINS,
            benchmark  = {'BTC': coin_dfs['BTC']},
            show       = False,
            cost       = c,
            save_html  = None,
        )
    cal = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0
    print(f'  {c:>8.4f} {mult:>5.1f}x {m["sharpe_ratio"]:>8.2f} '
          f'{m["total_return"]*100:>9.2f}% {m["max_drawdown"]*100:>9.2f}% {cal:>8.2f}')